In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.3 MLP feed-forward

> 🎯 **Goal:** Understand the second sublayer of a block, the position-wise MLP,
> and why it works on each token on its own.

**Why this matters.** A transformer block has two jobs, split between two sublayers.
Attention (earlier units) moves information *between* tokens. The MLP is the other
half: once a token has gathered context from its neighbors, the MLP lets it sit
and *think* about what it now knows. A large share of the model's parameters, and
much of its raw capacity, lives in these feed-forward layers.

**The intuition.** Attention is the group discussion where tokens swap notes. The
MLP is each token then going off to its own desk to process what it heard. No token
talks to any other during this step, each one transforms its own vector alone. That
private 'thinking' is where the model stores and applies a lot of its learned
knowledge.

**The mechanics.** The MLP is three steps applied to every position's vector: a
linear layer that **expands** the width (here `dim` to `4 * dim`, the standard 4x
blow-up), a **nonlinearity** (`GELU`, a smooth cousin of ReLU that lets the network
represent curved, non-linear functions), then a second linear that **projects**
back down to `dim` so the output shape matches the input and can be added to the
residual stream. The wide middle gives the layer room to compute richer features
before squeezing the result back to stream width. Crucially it is **position-wise**:
the exact same weights are applied independently to each token, with no mixing
across positions.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(1, 4, dim)
out = mlp(x)
print("MLP preserves shape:", tuple(out.shape))

**What just happened.** We fed a `(1, 4, 8)` batch through the MLP and got back the
same `(1, 4, 8)` shape: expand to 32, GELU, project back to 8. Shape preserved means
the output drops straight onto the residual stream.

The second cell proves the position-wise claim. We put one specific vector `same`
at position 2 of a sequence, then run that exact vector through the MLP all by
itself. The assert shows the two outputs are identical: the MLP gave `same` the
same answer whether it sat in a length-4 sequence or alone. The neighbors at other
positions had zero influence. The MLP genuinely never leaks information between
tokens, that mixing is attention's job and attention's alone.

⚠️ **Common pitfalls**
- Expecting the MLP to share information across tokens. It cannot; if your model
  needs context, that has to come from attention before the MLP runs.
- Dropping the nonlinearity. Two linear layers with nothing between them collapse
  into a single linear layer, the GELU is what makes the stack expressive.
- Treating the 4x expansion as magic. It is a tunable knob; bigger means more
  capacity and more compute, smaller means a leaner model.

🏋️ **Try it yourself.** Swap `nn.GELU()` for `nn.ReLU()` and rerun, the position-wise
property still holds. Then change `4 * dim` to `2 * dim` and to `8 * dim` and print
`sum(p.numel() for p in mlp.parameters())` each time to feel how the expansion
factor drives the parameter count.

In [ ]:
same = torch.randn(dim)
x2 = torch.randn(1, 4, dim)
x2[0, 2] = same
assert tuple(out.shape) == (1, 4, dim)
assert torch.allclose(mlp(x2)[0, 2], mlp(same.view(1, 1, dim))[0, 0], atol=1e-6)